In [34]:
import os
import numpy as np
import warnings 
import os
import xarray as xr
import numpy as np
import pandas as pd

from datetime import datetime

from string import Template

from experiment_configs import get_experiment_dict

In [35]:
class WindowedTCCCalculator:
    def __init__(self, data_dir, varname, experiment, ensemble_list, years, outdir,
                 reference_dir, reference_template, landmask_file=None, 
                 force_rewrite = False, windows = None, 
                 lat_bounds=None, lon_bounds=None, expected_frequency='daily'):
        self.data_dir = data_dir
        self.varname = varname
        self.experiment = experiment
        self.ensemble_list = ensemble_list
        self.years = years
        self.outdir = os.path.join(outdir, experiment)
        self.reference_dir = reference_dir
        self.reference_template = reference_template
        self.lat_bounds = lat_bounds
        self.lon_bounds = lon_bounds
        self.expected_frequency = expected_frequency
        self.region = "global"
        self.force_rewrite = force_rewrite
        self.windows = windows 
        
        self.Lv = 2.5e6
        self.landmask = xr.open_dataset(landmask_file)['landmask'] if landmask_file else None
        os.makedirs(self.outdir, exist_ok=True)
        
    def _convert_time_to_datetime64(self, ds):
        """
        Ensure the 'time' coordinate is in datetime64[ns] format.
        Converts from CFTimeIndex if necessary.
        """
        if "time" not in ds.coords:
            return ds  # nothing to do
    
        time_index = ds.indexes["time"]
    
        # Check if already datetime64
        if isinstance(time_index, pd.DatetimeIndex):
            return ds  # already in correct format
    
        try:
            # Attempt to convert CFTimeIndex to DatetimeIndex
            new_time = time_index.to_datetimeindex()
            ds["time"] = xr.DataArray(new_time.values, dims="time")
        except Exception as e:
            # Fallback: try coercing to string then to datetime
            print(f"[WARNING] Fallback time conversion: {e}")
            ds["time"] = pd.to_datetime(ds["time"].astype(str))
    
        return ds
    
    def _standardize_coords(self, da: xr.DataArray) -> xr.DataArray:
        # Rename coordinates if needed
        rename_dict = {}
        if 'latitude' in da.dims:
            rename_dict['latitude'] = 'lat'
        if 'longitude' in da.dims:
            rename_dict['longitude'] = 'lon'
        if rename_dict:
            da = da.rename(rename_dict)
    
        # Normalize longitude only if all values are >= 0 (assumes 0–360)
        if 'lon' in da.coords:
            lon_vals = da['lon'].values
            if (lon_vals >= 0).all():
                da = da.assign_coords(lon=((da.lon + 180) % 360 - 180))
                da = da.sortby('lon')
    
        return da

    def _load_prect(self,var,subvar1,subvar2,data_dir,alt_dir,ensemble,years):
        for base_dir in [data_dir, alt_dir]:
            file1 = [os.path.join(base_dir, f"{subvar1}.{ensemble}.{year}.nc") for year in self.years]
            file2 = [os.path.join(base_dir, f"{subvar2}.{ensemble}.{year}.nc") for year in self.years]
            print(f"[DEBUG] Trying: {file1}")
            print(f"[DEBUG] Trying: {file2}")
            if all(os.path.exists(f) for f in file1) and all(os.path.exists(f) for f in file2) :
                if base_dir == alt_dir:
                    self.expected_frequency = '6h'
                    print(f"[INFO] {v} found in 6-hourly path, will resample.")
                data1 = [xr.open_dataset(f) for f in file1]
                data2 = [xr.open_dataset(f) for f in file2]
                ds_merged1 = xr.concat(data1, dim='time')
                ds_merged1 = xr.decode_cf(ds_merged1)
                ds_merged2 = xr.concat(data2, dim='time')
                ds_merged2 = xr.decode_cf(ds_merged2)
                # Extract the variable after decoding
                ds = ds_merged1[v] 
                ds = ds + ds_merged2[v]
                ds.name = var
                
                ds = self._standardize_coords(ds)
                
                return ds
                    
    def _load_variable(self, varname, ensemble):
        original_var = varname
        if varname == "LHFLX":
            possible_vars = ["LHFLX", "QFLX"]     
        else:
            possible_vars = [varname] 
            
        alt_dir = self.data_dir.replace("daily", "6hourly")

        for v in possible_vars:
            for base_dir in [self.data_dir, alt_dir]:
                files = [os.path.join(base_dir, f"{v}.{ensemble}.{year}.nc") for year in self.years]
                print(f"[DEBUG] Trying: {files}")
                if all(os.path.exists(f) for f in files):
                    if base_dir == alt_dir:
                        self.expected_frequency = '6h'
                        print(f"[INFO] {v} found in 6-hourly path, will resample.")
                    data = [xr.open_dataset(f) for f in files]
                    ds_merged = xr.concat(data, dim='time')
                    ds_merged = self._convert_time_to_datetime64(ds_merged)

                    # Extract the variable after decoding
                    ds = ds_merged[v]
                    
                    if original_var == "LHFLX" and v == "QFLX":
                        ds = ds * self.Lv
                        ds.name = "LHFLX"
                        ds.attrs["units"] = "W/m2"
                    
                    ds = self._standardize_coords(ds)
                    if self.expected_frequency == 'daily':
                        ds['time'] = pd.to_datetime(ds['time'].values).normalize()
                    
                    return ds
                    
        if varname == 'PRECT':
            ds = self._load_prect(varname,'PRECC','PRECL',self.data_dir,alt_dir,ensemble,years)
        elif varname == 'PRECST':
            ds = self._load_prect(varname,'PRECSC','PRECSL',self.data_dir,alt_dir,ensemble,years)
            
        raise FileNotFoundError(f"[ERROR] Could not find {possible_vars} for {ensemble}.")

    def _load_reference(self, varname):
        ref_files = [
            os.path.join(self.reference_dir, 
                         self.reference_template.format(var=varname, year=year))
            for year in self.years
        ]
        print(f"[INFO] Looking for reference files: {ref_files}")
    
        missing = [f for f in ref_files if not os.path.exists(f)]
        if missing:
            raise FileNotFoundError(f"[ERROR] Missing reference files for {varname}: {missing}")
    
        ref_data = [xr.open_dataset(f)[varname] for f in ref_files]
        ref = xr.concat(ref_data, dim='time')
        ref = self._standardize_coords(ref)
        ref = self._convert_time_to_datetime64(ref)
        ref = ref.resample(time='1D').mean()
        ref['time'] = pd.to_datetime(ref['time'].values).normalize()
        
        return ref

    def _resample_if_needed(self, ds):
        if self.expected_frequency.lower() == '6h':
            ds = ds.resample(time='1D').mean()
            ds['time'] = pd.to_datetime(ds['time'].values).normalize()
        return ds
        
    def _subset_region(self, da):
        if self.lat_bounds:
            da = da.sel(lat=slice(*self.lat_bounds))
        if self.lon_bounds:
            da = da.sel(lon=slice(*self.lon_bounds))
        return da  # landmask not needed if region == global

    def _compute_tcc_for_window(self, model, reference, tslice) -> xr.DataArray:
        f = model.sel(time=tslice)
        o = reference.sel(time=tslice)
        f_anom = f - f.mean(dim='time')
        o_anom = o - o.mean(dim='time')
        num = (f_anom * o_anom).sum(dim='time')
        denom = np.sqrt((f_anom**2).sum(dim='time') * (o_anom**2).sum(dim='time'))
        return num / denom
        
    def _compute_rmse_for_window(self, model, reference, tslice) -> xr.DataArray:
        f = model.sel(time=tslice)
        o = reference.sel(time=tslice)
        mse = ((f - o) ** 2).mean(dim='time')
        rmse = np.sqrt(mse)
        return rmse
        
    def _save_outputs(self, label, tcc: xr.DataArray, rmse: xr.DataArray, ensemble, varname):
        fname = f"da_tcc_rmse_{self.experiment}_{ensemble}_{varname}_{label}.nc"
        path = os.path.join(self.outdir, fname)

        tcc.name = "TCC"
        tcc.attrs.update({
            "long_name": "Temporal Correlation Coefficient",
            "forecast_window": label
        })

        rmse.name = "RMSE"
        rmse.attrs.update({
            "long_name": "Root Mean Square Error",
            "forecast_window": label,
            "units": tcc.attrs.get("units", "unknown")  # use same units if available
        })

        combined = xr.Dataset({
            "TCC": tcc,
            "RMSE": rmse
        })
        if not os.path.exists(path) or self.force_rewrite:
            combined.to_netcdf(path)
            print(f"[SAVED] {path}")
    
    def process(self):
        varname = self.varname
    
        # Load and preprocess reference data
        print(f"[INFO] Loading reference data for {varname}")
        reference = self._load_reference(varname)
        reference = self._subset_region(reference)

        if reference is None or reference.sizes.get("time", 0) == 0:
            print(f"[WARNING] Reference data for {varname} is empty after preprocessing. Skipping.")
            return
    
        for ensemble in self.ensemble_list:
            print(f"[INFO] Processing {self.experiment} | {ensemble} | {varname}")
    
            # Load and preprocess model data                
            model = self._resample_if_needed(self._load_variable(varname, ensemble))
            model = self._subset_region(model)
            
            if model is None or model.sizes.get("time", 0) == 0:
                print(f"[WARNING] Model data for {ensemble} | {varname} is empty. Skipping.")
                continue
    
            # Loop through predefined time windows
            for label, (start_str, end_str) in self.windows.items():
                start = np.datetime64(start_str)
                end = np.datetime64(end_str)
                tslice = slice(start, end)
                try:
                    tcc = self._compute_tcc_for_window(model, reference, tslice)
                    rmse = self._compute_rmse_for_window(model, reference, tslice)
                    self._save_outputs(label, tcc, rmse, ensemble, varname)
                except Exception as e:
                    print(f"[ERROR] Failed to process window '{label}' for {ensemble} | {varname}: {e}")


In [36]:
if __name__ == "__main__":

    #top_path = "/compyfs/zhan391/v3_dart_cda_scratch"
    #out_path = "/compyfs/zhan391/v3_dart_cda_scratch/diag_dart"
    #fig_path = "/qfs/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"

    top_path = "/pscratch/sd/z/zhan391/e3sm_dart"
    out_path = "/pscratch/sd/z/zhan391/e3sm_dart/diag_dart"
    fig_path = "/people/zhan391/e3sm_dart_work/analysis/diagnostic/figures"
    
    landmask_file = os.path.join(out_path, "landmask_1x1.nc")
    
    exp = 'v3_spinup'
    years = [2011]
    exp_dict = get_experiment_dict(exp)  # Ensure this returns a valid experiment metadata dictionary
    windows = {
            #"D01-15":   ("2011-12-01", "2011-12-15"),
            #"D16-30":  ("2011-12-16", "2011-12-30"),
            "2012-12":  ("2011-12-01", "2011-12-31")
    }
    region = "global"

    ref_path = "/pscratch/sd/z/zhan391/e3sm_dart/Observations/ERA5/6hourly"
    ref_template = "ERA5.6hourly.en00.{var}.{year}01-{year}12.nc"
    varnames = {
        'TREFHT': "K",
        'TS'    : "K",
        'PRECT' : "W/m2",
        'PSL'   : 'Pa',
        'FLNS'  : "W/m2",
        'FSNS'  : "W/m2",
        'SHFLX' : "W/m2",
        'LHFLX' : "W/m2",
        'U200'  : 'm/s',
        'U850'  : 'm/s',
        'V200'  : 'm/s',
        'V850'  : 'm/s',
    }
    
    for var in varnames:
        print(f"\n[INFO] Processing variable: {var}")
        
        for exp_name, exp_meta in exp_dict.items():
            print(f"[INFO] Running experiment: {exp_name}")
            path = exp_meta['path']
            template_str = exp_meta['template']
            template = Template(template_str.replace('%(', '${').replace(')s', '}'))  # Convert to Python Template

            nens = exp_meta['nens']
            ensemble_list = [f"EN{n:02d}" for n in range(1, nens + 1)]

            processor = WindowedTCCCalculator(
                data_dir=path,
                varname=var,  # Single variable name
                experiment=exp_name,
                ensemble_list=ensemble_list,
                years=years,
                outdir=out_path,
                reference_dir=ref_path, 
                reference_template=ref_template,
                landmask_file=landmask_file,
                force_rewrite = True, 
                windows=windows
            )
            
            processor.process()



[INFO] Processing variable: TREFHT
[INFO] Running experiment: CTRLEN10
[INFO] Loading reference data for TREFHT
[INFO] Looking for reference files: ['/pscratch/sd/z/zhan391/e3sm_dart/Observations/ERA5/6hourly/ERA5.6hourly.en00.TREFHT.201101-201112.nc']
[INFO] Processing CTRLEN10 | EN01 | TREFHT
[DEBUG] Trying: ['/pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily/TREFHT.EN01.2011.nc']
[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/CTRLEN10/da_tcc_rmse_CTRLEN10_EN01_TREFHT_2012-12.nc
[INFO] Processing CTRLEN10 | EN02 | TREFHT
[DEBUG] Trying: ['/pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E3r5_compy/archive/post/atm/180x360_aave/ts/daily/TREFHT.EN02.2011.nc']
[SAVED] /pscratch/sd/z/zhan391/e3sm_dart/diag_dart/CTRLEN10/da_tcc_rmse_CTRLEN10_EN02_TREFHT_2012-12.nc
[INFO] Processing CTRLEN10 | EN03 | TREFHT
[DEBUG] Trying: ['/pscratch/sd/z/zhan391/e3sm_dart/CTRLEN10_F20TR_ne30pg2_r05_IcoswISC30E